# 🧭 Карта знаний R&D — извлечение на Colab (локальная LLM Qwen)

Обрабатывает документы из корпуса **без Yandex API**: поднимает Qwen2.5-7B через Ollama
на бесплатном GPU (T4) и гоняет стадию извлечения пайплайна. На выходе — те же
`data/processed/*.json`, что и от Yandex (формат чекпоинтов одинаков), так что позже
их можно грузить в граф хоть на Yandex-эмбеддингах, хоть на локальных (e5).

**Зачем Colab, а не только ноут:** T4 GPU быстрее CPU-инференса 7B-модели в 10–50×.
Извлечение — это тысячи LLM-вызовов, тут скорость решает. Чекпоинты пишутся на Google
Drive после каждого чанка, поэтому обрыв сессии Colab почти ничего не теряет.

**Порядок:** GPU → Ollama+Qwen → Drive → клон репо → извлечение → результат обратно на Drive.

## 1. Проверка GPU (нужен Runtime → Change runtime type → T4 GPU)

In [ ]:
!nvidia-smi -L || echo 'НЕТ GPU: Runtime -> Change runtime type -> T4 GPU'

## 2. Установка Ollama и запуск сервера

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request
# сервер в фоне; GPU Ollama подхватит сам
subprocess.Popen(["ollama", "serve"])
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        print("Ollama готов"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama не поднялся")

## 3. Модель Qwen2.5-7B + увеличенный контекст

`num_ctx 8192` обязателен: наш промпт (инструкция + чанк + до 4000 токенов JSON) не
влезает в дефолтное окно Ollama, иначе модель молча обрежет контекст.

In [ ]:
!ollama pull qwen2.5:7b-instruct

with open("Modelfile.qwen", "w") as f:
    f.write("FROM qwen2.5:7b-instruct\nPARAMETER num_ctx 8192\nPARAMETER temperature 0\n")
!ollama create minekg-qwen -f Modelfile.qwen
!ollama list

## 4. Google Drive — корпус и чекпоинты

Загрузи документы в Drive в папку `MyDrive/minekg/raw/` (можно сохранить структуру
подпапок: Доклады/, Обзоры/ ...). Результаты извлечения будут писаться в
`MyDrive/minekg/processed/` — переживают обрыв сессии.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
RAW = "/content/drive/MyDrive/minekg/raw"
PROC = "/content/drive/MyDrive/minekg/processed"
os.makedirs(RAW, exist_ok=True)
os.makedirs(PROC, exist_ok=True)
print("Документов в raw:", sum(len(fs) for _,_,fs in os.walk(RAW)))

## 5. Клон репозитория и зависимости

In [ ]:
REPO_URL = "https://github.com/0nt0n/nornicel.git"   # приватный? добавь токен: https://<TOKEN>@github.com/...
BRANCH = "feature/local_llm"

import os
if not os.path.exists("/content/nornicel"):
    !git clone --branch $BRANCH $REPO_URL /content/nornicel
%cd /content/nornicel
!pip install -q -r requirements.txt

## 6. Конфигурация: локальный бэкенд + пути на Drive

Переменные окружения выставляются ДО импорта проекта — `config.py` их подхватит
(load_dotenv не перетирает уже заданные переменные).

In [ ]:
import os
os.environ["LLM_BACKEND"] = "local"
os.environ["LOCAL_LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LOCAL_LLM_MODEL"] = "minekg-qwen"
os.environ["MAX_WORKERS"] = "4"          # на GPU можно параллелить запросы к Ollama
os.environ["RAW_DIR"] = RAW
os.environ["PROCESSED_DIR"] = PROC
# извлечение эмбеддинги НЕ трогает — ключи Yandex и EMBED_BACKEND тут не нужны
print("Бэкенд:", os.environ["LLM_BACKEND"], "| модель:", os.environ["LOCAL_LLM_MODEL"])

## 7. Смоук-тест: один вызов модели по нашему контракту

In [ ]:
import sys; sys.path.insert(0, "/content/nornicel")
from src.yandex import chat_json
from schema.ontology import EXTRACTION_JSON_SCHEMA
from src.extract.prompts import EXTRACT_SYSTEM, EXTRACT_USER_TMPL

demo = EXTRACT_USER_TMPL.format(lang="ru", doc_id="demo", page=1,
    text="Электроэкстракция никеля ведётся при плотности тока 250 А/м2 и температуре 60 C.")
out = chat_json(EXTRACT_SYSTEM, demo, "chunk_extraction", EXTRACTION_JSON_SCHEMA)
import json; print(json.dumps(out, ensure_ascii=False, indent=2)[:1500])

## 8. Извлечение

`--extract-only` — только LLM-стадия (граф/эмбеддинги не нужны). Ограничь подпапкой
через `--subdir`, чтобы делить корпус между Colab и ноутом. Уже обработанные файлы
пропускаются автоматически (чекпоинты на Drive).

In [ ]:
# весь корпус:  !python scripts/run_pipeline.py --extract-only
# одна подпапка:
!python scripts/run_pipeline.py --extract-only --subdir "Обзоры" 

## 9. Результат — архив на Drive (страховка)

In [ ]:
import shutil, datetime
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
dst = f"/content/drive/MyDrive/minekg/processed_backup_{stamp}"
shutil.make_archive(dst, "zip", PROC)
print("Готово:", dst + ".zip")
print("JSON-чекпоинтов:", len([f for f in os.listdir(PROC) if f.endswith('.json')]))

## Дальше

Скачай `MyDrive/minekg/processed/` к себе и положи в `data/processed/`. Когда вернётся
Yandex (или локально с `EMBED_BACKEND=e5`) — построй граф без повторного извлечения:

```bash
python scripts/run_pipeline.py --load-only
```

Уже загруженные чанки повторно не эмбеддятся.